## Data quality rules (production)
Fails invalid rows out to a quarantine table instead of silently dropping or letting them into silver.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "lena066636_silver")
dbutils.widgets.text("silver_table", "orders")
dbutils.widgets.text("quarantine_table", "orders_quarantine")


In [0]:
from pyspark.sql import functions as F

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
silver_table = f"{catalog}.{silver_schema}.{dbutils.widgets.get('silver_table')}"
quarantine_table = f"{catalog}.{silver_schema}.{dbutils.widgets.get('quarantine_table')}"


In [0]:
df = spark.table(silver_table)

rules = (
    F.col("order_id").isNotNull()
    & F.col("amount").isNotNull()
    & (F.col("amount").cast("double") >= 0.0)
)

df_valid = df.filter(rules)
df_invalid = df.filter(~rules)

In [0]:
df_valid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)

if df_invalid.limit(1).count() > 0:
    df_invalid.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(quarantine_table)
